In [20]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime, timedelta

import torch
import torch.nn as nn
import torch.optim as optim

from metpy.calc import dewpoint_from_specific_humidity
from metpy.units import units

import xesmf as xe

import os
import sys

sys.path.append('/home/548/cd3022/repos/solar-nowcast/modules')
import sat_preprocess
import clear_sky


# Set random seed for reproducibility
torch.manual_seed(42)

# Autodetect GPU and use if possible
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [21]:
# Time bounds to prepare data for
date = '2022-01-01'
num_days = '1'

start_year, start_month, start_day = [int(i) for i in date.split('-')]

# Data boundaries
start_date = datetime(start_year, start_month, start_day)
end_date   = datetime(start_year, start_month, start_day + int(num_days))

# Regional boundaries
lat_min=-35
lat_max=-28.5
lon_min=145
lon_max=151.5

In [22]:
# Get file paths for Heliosat datasets
base_path = Path('/g/data/rv74/satellite-products/arc/der/himawari-ahi/solar/p1s/latest')

files = []

current = start_date
while current < end_date:
    year  = current.year
    month = current.month
    day   = current.day

    file_path = base_path / f"{year}/{month:02d}/{day:02d}"
    
    if file_path.exists():  # important for missing days
        files.extend(file_path.rglob("*.nc"))
    
    current += timedelta(days=1)

In [23]:
# Select variables to be used from the heliosat dataset
helio_vars = [
    'surface_global_irradiance',
    'cloud_optical_depth',
    'solar_elevation',
]

# keep just the region and variables during preprocessing
def preprocess(ds):
    return ds.sel(
        latitude=slice(lat_min, lat_max),
        longitude=slice(lon_min, lon_max)
    )[helio_vars]


# Open the data
ghi = xr.open_mfdataset(files, preprocess=preprocess)

In [ ]:
# Himawari CTTH
base_path = Path('/g/data/rv74/satellite-products/arc/der/himawari-ahi/cloud/ctth/latest')

files = []

current = start_date
while current < end_date:
    year  = current.year
    month = current.month
    day   = current.day

    file_path = base_path / f"{year}/{month:02d}/{day:02d}"
    
    if file_path.exists():  # important for missing days
        files.extend(file_path.rglob("*.nc"))
    
    current += timedelta(days=1)
variables=['ctth_alti']

ctth = sat_preprocess.read_satproduct(
    files,
    variables,
    lat_min, lat_max, lon_min, lon_max,
    coords=True
)

regridder = xe.Regridder(
    ctth,
    ghi,
    method="bilinear",
    reuse_weights=False
)
ctth_interp = regridder(ctth)

# remove timezone for consistency with other datasets
ctth_interp["time"] = ctth_interp.indexes["time"].tz_localize(None)

In [25]:
%%time
rad_list = []

ch_list = [
    'B03',
    # 'B04',
    # 'B05',
    # 'B07',
    # 'B08',
    # 'B09',
    # 'B11',
    # 'B13',
    'B15'
]


for channel in ch_list:

    ds = sat_preprocess.read_himawari_channel(
        channel,
        start_date, end_date, # end_date,
        lat_min, lat_max, lon_min, lon_max,
        coords=True
    )

    rad_list.append(ds)
    print(f"{channel} DONE!")

# Use final radiance ds as reference for interpolation
# (if channels were listed in ascending order, should be the lowest resolution channel)
ref_ds = rad_list[-1]

interp_list = []

# interpolate radiance data to the same grid
for ds in rad_list:
    if ds.sizes != ref_ds.sizes:
        ds = ds.interp(y=ref_ds.y, x=ref_ds.x)

    # Drop conflicting coords (except for reference)
    if ds is not ref_ds:
        ds = ds.drop_vars(["latitude", "longitude"], errors="ignore")

    interp_list.append(ds)
    
# merge into single dataset
ds_rad = xr.merge(interp_list)

# Regrid radiances from (x, y) to (lat, lon)
regridder = xe.Regridder(
    ds_rad,
    ghi,
    method="bilinear",
    reuse_weights=False
)
ds_rad_interp = regridder(ds_rad)

B03 DONE!
B15 DONE!


<timed exec>:46: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
/opt/conda/envs/pet/lib/python3.11/site-packages/xarray/computation/apply_ufunc.py:450: PerformanceWarning: Regridding is increasing the number of chunks by a factor of 66.0, you might want to specify sizes in `output_chunks` in the regridder call. Default behaviour is to preserve the chunk sizes from the input (5, 323).
  result_vars[name] = func(*variable_args)
/opt/conda/envs/pet/lib/python3.11/site-packages/dask/array/routines.py:333: PerformanceWarning: Increasing number of chunks by factor of 73
  intermediate = blockwise(
/opt/conda/envs/pet/lib/python3.11/site-packages/xarray/computation/appl

CPU times: user 6min 37s, sys: 3min 8s, total: 9min 45s
Wall time: 10min 37s


In [36]:
# align all datasets so they can be merged
ghi_aligned, ctth_aligned, rad_aligned = xr.align(
    ghi,
    ctth_interp,
    ds_rad_interp,
    join="inner"
)

# merge datasets
ds = xr.merge([ghi_aligned, ctth_aligned, rad_aligned])

In [73]:
vars_to_save = [
    'surface_global_irradiance',
    'cloud_optical_depth',
    'solar_elevation',
    'ctth_alti',
    'channel_0003_scaled_radiance',
    'channel_0015_brightness_temperature',
    
]


output_dir = "/scratch/er8/cd3022/CPDiT_data"
os.makedirs(output_dir, exist_ok=True)

for time in ds.time:
    
    t=np.datetime_as_string(time.values)
    dt = t.astype("datetime64[ms]").astype(object)
    # break
    formatted = dt.strftime("%Y%m%d%H%M")

    # Collect variables
    arrays = []

    for var in vars_to_save:
        data = ds[var].sel(time=time).values
        arrays.append(data)

    # Shape: (n_vars, lat, lon)
    stacked = np.stack(arrays, axis=0)

    tensor = torch.from_numpy(stacked.astype(np.float32))

    torch.save(
        {
            "data": tensor,
            "variables": vars_to_save,
        },
        os.path.join(output_dir, f"{formatted}.pt"),
    )

In [67]:
prepare = petpipe.Pipeline(
    satpipe,
    # GeospatialTimeSeriesMerge(reference_dataset=satpipe['2021-06-09T02']), # These are pretty similar grids, so just pick one
    petdata.transform.region.Bounding(-36, -31, 145, 151),
    iterator=petpipe.iterators.DateRange(2021, 2023, interval='10 minutes')
)
ipipe = iter(prepare)

output_dir = '/scratch/nf33/cd3022/pyearth'
for i in range(0, 1):

    #get time label
    t_X=np.datetime_as_string(ds['time'].values[0])
    dt_X = t.astype('datetime64[ms]').astype(object)
    formatted = dt.strftime('%Y%m%d%H%M')
    # torch.save(data, file)

In [71]:
dt_X

datetime.datetime(2022, 1, 1, 0, 0)